<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/projects/tutorials/03-build-a-team-with-langgraph.ipynb)

# Build a team with LangGraph

**Goal:** define a team of narrow roles in code you can point at, wire them into a graph, draw
that graph, and watch one question move through it node by node.

Project 03 imports `build_team` and measures what it costs. This tutorial is the slow half: the
state, the roles, one node at a time with the prompt it sends printed right above it, the
wiring, the picture, and the stream. Nothing here is graded.

**It runs with no model, no key and no LangGraph.** The setup cell prints `[live]` and the model
name when a local model is running, or `[recorded]` and a date when it replays one real run from
`projects/tutorials/fixtures/03-build-a-team-with-langgraph.json`. The graph sections print an
install line and carry on when the optional extra is missing:

```bash
uv sync --extra projects --extra agents
```

Name both extras. `uv sync` makes the environment match exactly, so asking for `agents` alone
uninstalls `projects`.

It uses the course corpus in `data/corpus/` and the retrieval in
`src/bootcamp_agent/retrieval.py`. [Tutorial 1](01-calling-a-model.ipynb) covers the model call
underneath, and [tutorial 2](02-tools-with-limits.ipynb) covers the tools.

## Setup

Run these two cells first. The first line printed tells you the lane.

In [ ]:
# Setup: find the course, pick the lane, and print which one you are on.
import json
import operator
import os
import sys
import urllib.request
from dataclasses import dataclass
from pathlib import Path
from typing import Annotated, TypedDict

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    # Colab starts in /content with no course in it, so fetch the public copy once.
    import subprocess

    ROOT = Path("/content/dev3pack")
    if not (ROOT / "pyproject.toml").exists():
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1",
             "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(ROOT)],
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "langgraph", "grandalf"],
                   check=False)
sys.path.insert(0, str(ROOT / "src"))

from bootcamp_agent.llm import FakeLLM
from bootcamp_agent.ollama import DEFAULT_BASE_URL, DEFAULT_MODEL, OllamaClient, probe

MODEL = DEFAULT_MODEL  # the recording was made with this model, so live uses it too
BASE_URL = os.environ.get("OLLAMA_BASE_URL") or DEFAULT_BASE_URL  # the /v1 address
NATIVE_URL = BASE_URL.removesuffix("/v1")  # Ollama's own API lives one level up
FIXTURE = ROOT / "projects" / "tutorials" / "fixtures" / "03-build-a-team-with-langgraph.json"
RECORDED = json.loads(FIXTURE.read_text(encoding="utf-8"))
CHECK = probe(MODEL, BASE_URL)
LIVE = CHECK.ok

if LIVE:
    print(f"[live] {MODEL}")
else:
    print(f"[recorded] {RECORDED['_provenance']['recorded']}")
    print(f"  replays one real run of {RECORDED['_provenance']['model']}. To go live: {CHECK.fix}")

# The optional extra, imported once and guarded once. Every graph cell below reads
# LANGGRAPH and says what it could not do, rather than raising.
try:
    from langgraph.graph import END, START, StateGraph

    LANGGRAPH = True
    print("[langgraph] installed: sections 4 and 5 will build and draw the graph")
except ImportError:
    LANGGRAPH = False
    print("[langgraph] not installed: uv sync --extra projects --extra agents")
    print("            sections 4 and 5 print what they could not do; the rest runs.")

In [ ]:
# Helpers: record every reply when live, replay the recording when not.
REPLIES = {}  # user prompt -> model reply, filled as you run
RAW = {}  # label -> {"request": ..., "response": ...} for calls to Ollama's own API
NOT_RECORDED = "(not in the recording: start Ollama and run live to ask something new)"


class Recorder:
    """Wraps any LLMClient and keeps each reply, keyed by the user prompt."""

    def __init__(self, inner):
        self.inner = inner

    def complete(self, system, user):
        reply = self.inner.complete(system=system, user=user)
        # Keep the first reply: a later cell that asks the same prompt replays the same one.
        REPLIES.setdefault(user, reply)
        return reply


def replay(replies):
    # Longest prompt first: a follow-up prompt can contain an earlier one, so it must match first.
    ordered = dict(sorted(replies.items(), key=lambda item: -len(item[0])))
    return FakeLLM(responses=ordered, default=NOT_RECORDED)


def ollama_post(label, path, body, live=LIVE):
    """POST to Ollama's own API when live. Replay the recorded response when not."""
    if not live:
        recorded = RECORDED["raw"][label]
        if recorded["request"] != body:
            raise LookupError(f"{label}: this request is not the recorded one. Start Ollama to send it.")
        return recorded["response"]
    request = urllib.request.Request(
        NATIVE_URL + path,
        data=json.dumps(body).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=120) as response:
        payload = json.loads(response.read().decode("utf-8"))
    RAW.setdefault(label, {"request": body, "response": payload})
    return payload


llm = Recorder(OllamaClient(model=MODEL, base_url=BASE_URL) if LIVE else replay(RECORDED["replies"]))

from bootcamp_agent.documents import load_corpus
from bootcamp_agent.retrieval import retrieve
from bootcamp_agent.schema import (
    ANSWER_JSON_INSTRUCTIONS,
    AnswerParseError,
    ResearchAnswer,
    parse_research_answer,
)

DOCS = load_corpus(ROOT / "data" / "corpus")
QUESTIONS = {
    "chunking": "How do I split text into chunks for retrieval?",
    "injection": "What is prompt injection and how do I defend against it?",
}
print(f"{len(DOCS)} documents, {len(QUESTIONS)} questions")

## 1. The state, and the reducer that lets two nodes write to one list

A team needs one place to keep what it knows. In LangGraph that place is a `TypedDict`, and
every node reads it and returns **only the fields it changed**. The runner merges that patch.

Merging is where the surprise lives. By default a field in a patch **replaces** the field in the
state. That is what you want for `answer`, and it is wrong for `calls`: the critic returning
`{"calls": ["critic"]}` would throw the writer's entry away, and your bill would read 1 where
you spent 2.

`Annotated[list, operator.add]` says "when two patches touch this field, add them" instead. One
line, and it is the difference between a log and the last line of a log.

**What to look at:**

- Two fields, two behaviours, in one class. `answer` replaces. `calls` appends.
- The merge demonstration below uses no framework at all. It is two dicts and a rule, because
  that is all a reducer is.
- The comment on `passages`. It replaces on purpose: a second researcher should not silently
  double the evidence, and if you ever want two of them, that is the line you change.

In [ ]:
# The state: one dict the whole team reads, with one field that appends instead of replacing.
class TeamState(TypedDict, total=False):
    question: str
    passages: list  # replaced: one researcher fetches, and re-fetching should not double it
    draft: object  # replaced: there is one current draft, and the newest one is it
    critique: str  # replaced
    approved: bool  # replaced
    revisions: int  # replaced
    max_revisions: int  # replaced
    stopped_because: str  # replaced
    calls: Annotated[list, operator.add]  # APPENDED: every node adds its own, none overwrites


for field, kind in TeamState.__annotations__.items():
    rule = "append" if "Annotated" in str(kind) else "replace"
    print(f"{field:16} {rule}")

In [ ]:
# What the two rules do, with no framework in the room. A reducer is this and nothing more.
def merge(state: dict, patch: dict, appended: set) -> dict:
    """Merge a node's patch into the state. Fields in `appended` add; the rest replace."""
    merged = dict(state)
    for key, value in patch.items():
        merged[key] = state.get(key, []) + value if key in appended else value
    return merged


state = {"calls": ["writer"], "answer": "a first draft"}
patch = {"calls": ["critic"], "answer": "a second draft"}

print("no reducer anywhere: ", merge(state, patch, appended=set()))
print("calls has a reducer: ", merge(state, patch, appended={"calls"}))
print("\nThe first line says the team made one call. It made two. Nothing raised, and the")
print("number you would have reported is simply wrong.")

In [ ]:
# Try it: add a "notes" field that appends, and have two nodes write to it.
notes_state = {"notes": ["the coordinator saw a question about chunking"]}
notes_patch = {"notes": ["the researcher found 3 passages"]}
print("replace:", merge(notes_state, notes_patch, appended=set()))
print("append: ", merge(notes_state, notes_patch, appended={"notes"}))

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain what `Annotated[list, operator.add]` does to a LangGraph state field in section 1 of projects/tutorials/03-build-a-team-with-langgraph.ipynb. Do not change the code."
> - "Show me one bug that a missing reducer would cause in a team that counts its own model calls. Do not change the code."

## 2. The team, as data

"Define the team" should be a thing you can point at, not a paragraph. So the team is a list of
four small objects, and each one carries everything that makes a role a role:

| Field | What it is |
|---|---|
| `name` | the node name in the graph, and the entry in `calls` |
| `job` | one sentence. If it needs two, the role is doing two jobs |
| `tools` | what it may reach. Most roles get none, and that is the design |
| `calls_model` | whether it spends a call. Two of the four do |
| `prompt` | the exact words it sends, or empty when it sends none |

Read the table the cell prints and you have read the team. Add a fifth role and the table grows
by one line, which is the test of whether "the team is data" is true or just said.

**What to look at:**

- Two roles have `calls_model=False`. They are the cheap half, and they do the routing and the
  fetching.
- Three roles have no tools. Section 6 of project 03's notebook says why the writer's row is
  empty, and it is the same reason here.
- The prompt column is short because the prompts live next to their nodes in section 3, where
  you can read them whole.

In [ ]:
# The team, as four objects. This is the answer to "how do I define the team?".
@dataclass(frozen=True)
class Role:
    """One member of the team. The graph node is named after it, and so is its entry in calls."""

    name: str
    job: str
    tools: tuple
    calls_model: bool


TEAM = [
    Role("coordinator", "Read the question and set the budget.", (), False),
    Role("researcher", "Fetch passages from the corpus, and nothing else.", ("retrieve",), False),
    Role("writer", "Write one answer as strict JSON, citing only what came back.", (), True),
    Role("critic", "Approve the draft, or name one fix. Never rewrite it.", (), True),
]

print(f"{'role':13} {'model':6} {'tools':10} job")
for role in TEAM:
    print(f"{role.name:13} {str(role.calls_model):6} {', '.join(role.tools) or '-':10} {role.job}")
print(f"\nmodel calls per pass: {sum(1 for role in TEAM if role.calls_model)}")

In [ ]:
# Try it: add a fact-checker to the team and read what it costs before you build it.
PROPOSED = [*TEAM, Role("fact_checker", "Check every citation against its source.", (), True)]
print(f"roles: {len(TEAM)} -> {len(PROPOSED)}")
print(f"model calls per pass: {sum(1 for r in TEAM if r.calls_model)} -> "
      f"{sum(1 for r in PROPOSED if r.calls_model)}")
print("\nA role is not free. Write down what the extra call buys before you add the node.")

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain why the team in section 2 of projects/tutorials/03-build-a-team-with-langgraph.ipynb is a list of dataclasses instead of four functions. Do not change the code."
> - "Three of the four roles have no tools. Explain what each one would be able to do wrong if it had one. Do not change the code."

## 3. One node, with the prompt right above it

Here is the layout worth copying, and it is the whole point of this section: **the words the
model receives sit immediately above the code that sends them.** A prompt hidden two files away
is a prompt nobody reads, and nobody can review what they never read.

So each node below is written as a pair:

1. a module-level string with named `{placeholders}`, and
2. a short function that formats it, calls the model once, and returns a patch.

A node is small on purpose. If a node needs scrolling, it is doing two jobs.

**What to look at:**

- The writer's system prompt is `ANSWER_JSON_INSTRUCTIONS`, the same contract session 3 built.
  Its user message is a template you can read start to finish.
- The critic's `APPROVE`. The code matches it with `.upper().startswith("APPROVE")`, so that
  token is not a word in a sentence. It is the interface. Reword the prompt without it and the
  run changes shape while nothing errors.
- The last cell prints the exact HTTP body the writer sends, keys and all. Everything above is
  string formatting on top of that one POST.

In [ ]:
# The coordinator and the researcher. No prompts here, because they call no model.
BUDGET = 6
TOP_K = 3


def coordinator(state: dict) -> dict:
    """Set the budget. It calls nothing, so it is the cheapest role on the team."""
    return {"max_revisions": state.get("max_revisions", 1), "calls": []}


def researcher(state: dict) -> dict:
    """Fetch passages, and only that. Nothing found is an exit, not a crash."""
    found = retrieve(state["question"], DOCS, top_k=TOP_K)
    if not found:
        return {
            "passages": [],
            "stopped_because": "answered",
            "draft": ResearchAnswer("Nothing in the corpus answers this.", (), 0.0, True),
        }
    return {"passages": [(round(s.score, 2), s.chunk.doc_id, s.chunk.text) for s in found]}


print(coordinator({"question": QUESTIONS["chunking"]}))
for score, doc_id, text in researcher({"question": QUESTIONS["chunking"]})["passages"]:
    print(f"{score:6}  {doc_id:18} {text[:62]}…")

In [ ]:
# The writer: its prompt first, then the four lines that send it.
WRITER_PROMPT = """Write the answer.
Question: {question}

Passages:
{passages}
{fix}"""


def format_passages(passages: list) -> str:
    """Untrusted text, capped before a model sees it. Session 4's rule."""
    return "\n\n".join(f"[{doc_id}] (score {score})\n{text[:900]}"
                        for score, doc_id, text in passages)


def writer(state: dict) -> dict:
    """ONE model call, strict JSON, and citations only for what the researcher returned."""
    if len(state["calls"]) >= BUDGET:
        return {"stopped_because": "budget",
                "draft": ResearchAnswer("The budget ran out before the writer ran.", (), 0.0, True)}
    critique = state.get("critique", "")
    message = WRITER_PROMPT.format(
        question=state["question"],
        passages=format_passages(state["passages"]),
        fix=f"\nOne fix the reviewer asked for: {critique[:300]}" if critique else "",
    )
    raw = llm.complete(system=ANSWER_JSON_INSTRUCTIONS, user=message)
    revisions = state.get("revisions", 0) + (1 if critique else 0)
    try:
        answer = parse_research_answer(raw)
    except AnswerParseError as error:
        # Session 3's rule, as one branch: prose is a failed step, never an answer.
        return {"calls": ["writer"], "revisions": revisions, "stopped_because": "tool_error",
                "draft": ResearchAnswer(f"The reply did not parse: {error}", (), 0.0, True)}
    retrieved = {doc_id for _, doc_id, _ in state["passages"]}
    kept = tuple(cited for cited in answer.citations if cited in retrieved)
    return {
        "calls": ["writer"],
        "revisions": revisions,
        "draft": ResearchAnswer(answer.answer, kept, answer.confidence,
                                answer.needs_human_review or len(kept) != len(answer.citations)),
    }


print(WRITER_PROMPT.format(question=QUESTIONS["chunking"], passages="[rag-basics] …", fix=""))

In [ ]:
# The critic: its prompt first, then the run. One word in it is the interface.
CRITIC_RULES = (
    "You are a strict reviewer. Reply APPROVE if the answer is supported by the "
    "passages and cites them. Otherwise name one concrete fix in one sentence. "
    "Never rewrite the answer yourself."
)

CRITIC_PROMPT = """Review the draft answer.
Question: {question}

Draft: {draft}
Cited: {cited}

Passages:
{passages}"""


def critic(state: dict) -> dict:
    """ONE model call: approve, or ask for one revision. The cap lives here."""
    if len(state["calls"]) >= BUDGET:
        return {"stopped_because": "budget"}
    draft = state["draft"]
    message = CRITIC_PROMPT.format(
        question=state["question"],
        draft=draft.answer[:1200],
        cited=", ".join(draft.citations) or "nothing",
        passages=format_passages(state["passages"]),
    )
    verdict = llm.complete(system=CRITIC_RULES, user=message).strip()
    # The interface is this string test, not the sentence around it. "Looks good to me"
    # approves the draft in English and rejects it here, and the run pays for a revision.
    approved = verdict.upper().startswith("APPROVE")
    patch = {"calls": ["critic"], "critique": verdict, "approved": approved}
    if approved:
        patch["stopped_because"] = "answered"
    elif state.get("revisions", 0) >= state.get("max_revisions", 1):
        patch["stopped_because"] = "budget"
    return patch


print(CRITIC_RULES)
print()
for verdict in ("APPROVE", "Approved.", "Looks good to me.", "LGTM"):
    print(f"{verdict.upper().startswith('APPROVE')!s:5}  {verdict!r}")

In [ ]:
# The HTTP underneath. Everything above is string formatting on top of this one POST.
sample = researcher({"question": QUESTIONS["chunking"]})["passages"]
body = {
    "model": MODEL,
    "messages": [
        {"role": "system", "content": ANSWER_JSON_INSTRUCTIONS},
        {"role": "user", "content": WRITER_PROMPT.format(
            question=QUESTIONS["chunking"], passages=format_passages(sample), fix="")},
    ],
    "stream": False,
}
print(json.dumps({**body, "messages": [{**m, "content": m["content"][:110] + "…"}
                                       for m in body["messages"]]}, indent=1))
payload = ollama_post("writer-raw", "/api/chat", body)
print("\nthe reply, as the server sent it:")
print(payload["message"]["content"][:300])

In [ ]:
# Try it: change the writer's prompt and read the message before you send it.
MY_PROMPT = WRITER_PROMPT.replace("Write the answer.", "Answer in one sentence.")
print(MY_PROMPT.format(question=QUESTIONS["injection"], passages="[prompt-injection] …", fix="")[:300])
print("\nOn the recorded lane a changed prompt matches nothing, so the replay answers the")
print("default. That is the honest outcome: nothing was recorded, so nothing is known.")

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain the prompt-then-node layout in section 3 of projects/tutorials/03-build-a-team-with-langgraph.ipynb, and why the prompt is a module level string. Do not change the code."
> - "Which word in the critic's prompt does the code read, and what happens to the run if I reword the prompt without it? Do not change the code."

## 4. Wire it, compile it, and draw it

Four nodes and a picture. The wiring is five kinds of line and nothing else:

| Line | What it does |
|---|---|
| `StateGraph(TeamState)` | declares which dict every node reads and patches |
| `add_node(name, fn)` | one role, one function |
| `add_edge(START, "coordinator")` | where a run begins |
| `add_edge("coordinator", "researcher")` | an edge that always fires |
| `add_conditional_edges(node, decide, targets)` | an edge that asks a function first |

`decide` is the only routing logic in the whole team, and it is one line: **a state with
`stopped_because` set is finished.** Every edge asks that, so "what happens after a parse
failure" is a line you can point at rather than a branch you have to find.

Then `compile()`, and then the part the founder asked for out loud: **draw it.** Three rungs,
because each one can be missing:

1. `draw_mermaid_png()` is the prettiest and posts to mermaid.ink, so it needs the network.
2. `draw_ascii()` works offline and needs `grandalf`, which the `agents` extra installs.
3. `draw_mermaid()` returns the diagram as text and always works.

**What to look at:**

- The picture, and the one edge that goes backwards: `critic -> writer`. That is the revision,
  and it is the reason a team can cost four calls instead of two.
- `add_conditional_edges` naming every target it may return. LangGraph checks that list, so a
  typo in `decide` is caught when the graph is built rather than when it runs.
- Which of the three drawing rungs your machine used. All three are correct outcomes.

In [ ]:
# Wire the four nodes, compile, and draw. Guarded: without LangGraph this cell says so.
NEXT = {"coordinator": "researcher", "researcher": "writer", "writer": "critic",
        "critic": "writer"}
NODES = {"coordinator": coordinator, "researcher": researcher, "writer": writer, "critic": critic}

graph = None
if not LANGGRAPH:
    print("langgraph is not installed, so there is no graph to build or draw.")
    print("  uv sync --extra projects --extra agents")
else:

    def decide_from(node: str):
        """One rule, asked on every edge: a state with stopped_because set is finished."""

        def decide(state: dict) -> str:
            return END if state.get("stopped_because") else NEXT[node]

        return decide

    builder = StateGraph(TeamState)
    for name, function in NODES.items():
        builder.add_node(name, function)
    builder.add_edge(START, "coordinator")
    for name, target in NEXT.items():
        # The third argument lists every target this edge may return. LangGraph checks
        # it, so a typo in `decide` is caught when the graph is built, not when it runs.
        builder.add_conditional_edges(name, decide_from(name), {target: target, END: END})
    graph = builder.compile()
    print(f"compiled: {len(NODES)} nodes, entry point 'coordinator'")

In [ ]:
# The picture. Three rungs, and each one can be the missing one on your machine.
def can_reach_mermaid(timeout: float = 2.0) -> bool:
    """A two second look before a long wait: draw_mermaid_png posts to mermaid.ink."""
    import socket

    try:
        socket.create_connection(("mermaid.ink", 443), timeout=timeout).close()
        return True
    except OSError:
        return False


def draw(compiled) -> None:
    """Draw the graph the best way this machine can, and say which way that was."""
    drawable = compiled.get_graph()
    try:
        if not can_reach_mermaid():
            raise OSError("mermaid.ink is not reachable")
        from IPython.display import Image, display

        display(Image(drawable.draw_mermaid_png()))
        print("drawn with draw_mermaid_png (it posted to mermaid.ink, so it used the network)")
        return
    except Exception as error:  # no network, or no renderer: both are ordinary
        print(f"no PNG ({type(error).__name__}), falling back")
    try:
        print(drawable.draw_ascii())
        print("drawn with draw_ascii (offline, via grandalf)")
        return
    except ImportError:
        print("no grandalf either, so here is the diagram as text")
    print(drawable.draw_mermaid())


if graph is None:
    print("no graph on this machine: uv sync --extra projects --extra agents")
else:
    draw(graph)

In [ ]:
# Try it: add a fifth node and draw it again. The picture changes; the roles do not.
if graph is None:
    print("needs langgraph: uv sync --extra projects --extra agents")
else:
    trial = StateGraph(TeamState)
    for name, function in NODES.items():
        trial.add_node(name, function)
    trial.add_node("archivist", lambda state: {"calls": []})  # a node that does nothing yet
    trial.add_edge(START, "coordinator")
    trial.add_edge("coordinator", "researcher")
    trial.add_edge("researcher", "writer")
    trial.add_edge("writer", "critic")
    trial.add_edge("critic", "archivist")
    trial.add_edge("archivist", END)
    print(trial.compile().get_graph().draw_mermaid())

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain each LangGraph call in section 4 of projects/tutorials/03-build-a-team-with-langgraph.ipynb: StateGraph, add_node, add_edge, add_conditional_edges and compile. Do not change the code."
> - "Which edge in the drawn graph goes backwards, and what does that edge cost in model calls? Do not change the code."

## 5. Watch it run

`graph.invoke(state)` gives you the finished dict and tells you nothing about how it got there.
`graph.stream(state, stream_mode=...)` gives you the run as it happens, and the mode decides
what each step hands you:

| Mode | What one step is | Use it to |
|---|---|---|
| `"updates"` | `{node_name: the patch it returned}` | see which node fired and what it changed |
| `"values"` | the whole state after that node | watch one field fill in, step by step |

`updates` is the one to reach for first. It is short, it names the node, and a run that goes
somewhere you did not expect shows you the wrong node by name instead of a final dict that is
merely odd.

**What to look at:**

- The order: coordinator, researcher, writer, critic. Two of those printed nothing under
  `calls`, because they spent nothing.
- `calls` growing by one entry at a time under `values`. That is section 1's reducer, doing the
  only thing it does.
- Whether the critic approved. If it did, the run stops at four nodes. If it did not, the writer
  runs again and you watch the backwards edge fire.

In [ ]:
# stream_mode="updates": one line per node, and only what that node changed.
START_STATE = {"question": QUESTIONS["chunking"], "calls": [], "revisions": 0, "max_revisions": 1}

if graph is None:
    print("needs langgraph: uv sync --extra projects --extra agents")
else:
    for step, event in enumerate(graph.stream(dict(START_STATE), stream_mode="updates"), start=1):
        for node, patch in event.items():
            print(f"{step}. {node:12} changed {sorted(patch)}")
            if "passages" in patch:
                print(f"   {len(patch['passages'])} passages fetched")
            if "draft" in patch:
                print(f"   draft: {patch['draft'].answer[:92]}")
            if "critique" in patch:
                print(f"   verdict: {patch['critique'].splitlines()[0][:92]}")

In [ ]:
# stream_mode="values": the whole state after each node, so you watch one field fill in.
if graph is None:
    print("needs langgraph: uv sync --extra projects --extra agents")
else:
    # The first event is the state you handed in, before any node ran. So there are five
    # lines for four nodes, and line 1 is your input rather than a node's work.
    for step, state in enumerate(graph.stream(dict(START_STATE), stream_mode="values")):
        label = "input" if step == 0 else f"{step}."
        print(f"{label:6} calls={state.get('calls', [])} "
              f"passages={len(state.get('passages', []))} "
              f"approved={state.get('approved')} stopped={state.get('stopped_because')}")
    print("\n`calls` grew one entry at a time. Without section 1's reducer the last line would")
    print("read ['critic'], and the run would look half as expensive as it was.")

In [ ]:
# Try it: ask the other question and watch which node the run ends on.
if graph is None:
    print("needs langgraph: uv sync --extra projects --extra agents")
else:
    MINE = QUESTIONS["injection"]
    last = None
    for event in graph.stream({"question": MINE, "calls": [], "revisions": 0, "max_revisions": 1},
                              stream_mode="updates"):
        last = event
    node, patch = next(iter(last.items()))
    print(f"last node: {node}, stopped_because={patch.get('stopped_because')}")

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain the difference between stream_mode 'updates' and 'values' in section 5 of projects/tutorials/03-build-a-team-with-langgraph.ipynb, and when I would want each. Do not change the code."
> - "From the streamed run, tell me which nodes spent a model call and which did not. Do not change the code."

## 6. Who validates: your parser, or the framework

LangChain offers `llm.with_structured_output(SomeModel)`. You hand it a schema, it hands you an
object, and the parsing step disappears from your code. It is genuinely convenient.

**This course does the opposite, and session 3 is built on it: the application validates, never
the model.** The writer in section 3 gets a plain string back and runs it through
`parse_research_answer`, and a reply that does not parse becomes a flagged refusal with
`stopped_because="tool_error"`.

The difference is not style. `with_structured_output` is convenient right up to the day it
returns something plausible and wrong: the shape is valid, the fields are filled, and nothing in
your code ever saw the text the model actually sent. When you own the parse, the failure is a
message that names the field, and you decide what happens next. When the framework owns it, you
find out downstream.

Use whichever you like in your own work. Know which one you chose.

**Where this goes next.** The reference this tutorial borrows its layout from fans several
interviews out in parallel with LangGraph's `Send()`, which is a good technique and a different
lesson. Build a team you can read first.

**What to look at:**

- Four replies through `parse_research_answer`, and the exact message each one gets.
- The one that is valid JSON, correctly typed, and about the wrong thing. The parser passes it,
  because a parser checks shape and nothing else. The citation check is what catches that one,
  and it is three lines in section 3's writer.

In [ ]:
# Four replies, one parser. Read the message each one gets.
CANDIDATES = {
    "the contract": '{"answer": "Chunking splits documents.", "citations": ["rag-basics"], '
                    '"confidence": 0.8, "needs_human_review": false}',
    "prose, no JSON": "Chunking splits documents into passages before you embed them.",
    "a field it invented": '{"answer": "ok", "citations": [], "confidence": 0.1, '
                           '"needs_human_review": true, "source": "rag-basics"}',
    "valid, and wrong": '{"answer": "Chunking is a kind of cheese.", "citations": ["rag-basics"], '
                        '"confidence": 0.99, "needs_human_review": false}',
}
for label, raw in CANDIDATES.items():
    try:
        answer = parse_research_answer(raw)
        print(f"{label:22} parsed: {answer.answer[:52]}")
    except AnswerParseError as error:
        print(f"{label:22} refused: {error}")
print("\nThe last one parsed. A parser checks SHAPE. Whether the answer is true is a different")
print("job, and in this team the citation check does the part of it a machine can do.")

In [ ]:
# The citation check: three lines, and the only reason the last reply above is catchable.
passages = researcher({"question": QUESTIONS["chunking"]})["passages"]
retrieved = {doc_id for _, doc_id, _ in passages}
for label in ("the contract", "valid, and wrong"):
    cited = parse_research_answer(CANDIDATES[label]).citations
    invented = [c for c in cited if c not in retrieved]
    print(f"{label:22} cited={list(cited)} invented={invented or 'none'}")
print(f"\nretrieved: {sorted(retrieved)}")
print("Both cite a real document, so both pass. The check bounds where an answer may come")
print("from; it does not read it. That is honest about what it is, which is the point.")

In [ ]:
# Try it: write a reply that passes the parser and should not pass a human.
MINE = ('{"answer": "Prompt injection is solved by asking the model nicely.", '
        '"citations": ["prompt-injection"], "confidence": 1.0, "needs_human_review": false}')
answer = parse_research_answer(MINE)
print(f"parsed:     {answer.answer}")
print(f"confidence: {answer.confidence}, flagged: {answer.needs_human_review}")
print("\nNothing in this notebook catches that one. The critic in section 3 is the role whose")
print("job it is, and section 5 shows it costing a model call to do it.")

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain why section 6 of projects/tutorials/03-build-a-team-with-langgraph.ipynb parses the reply itself instead of using with_structured_output. Do not change the code."
> - "Give me one failure that with_structured_output would hide and my own parser would surface. Do not change the code."

In [ ]:
# Save this run as the recording. Maintainers only: it does nothing unless TUTORIAL_RECORD=1.
from datetime import date

if LIVE and os.environ.get("TUTORIAL_RECORD") == "1":
    RECORDED = {
        "_provenance": {
            "recorded": date.today().isoformat(),
            "model": MODEL,
            "lane": "one real run of the local model, replayed when no model is running",
            "auth_sent": "none",
            "temperature": "not set: the client sends none, so Ollama used its default and replies vary run to run",
            "keys": "replies are keyed by the full user prompt, passages included; raw calls by label, with the exact request",
            "is_evidence_of": "what this model returned for these prompts on that run, byte for byte, including whether the critic approved",
            "is_not_evidence_of": "what it returns every time, nor that the answers are right. A 7B model words things differently on every run and nobody graded the content. Run it live to see yours.",
        },
        "replies": REPLIES,
        "raw": RAW,
    }
    FIXTURE.write_text(json.dumps(RECORDED, indent=1, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"wrote {FIXTURE.relative_to(ROOT)}: {len(REPLIES)} replies, {len(RAW)} raw calls")
else:
    print("nothing saved: this cell writes only when live and TUTORIAL_RECORD=1")

## Resources

The course pages and code this tutorial draws on:

- [Session 8: loops and graphs](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit2/session-08-loops-and-graphs/introduction.mdx)
- [Session 8: subagents in a graph](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit2/session-08-loops-and-graphs/langgraph_subagents.py)
- [Session 3: structured outputs](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-03-structured-outputs/introduction.mdx)
- [Session 4: bounded tools](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit1/session-04-bounded-tools/introduction.mdx)
- [The team this tutorial is the slow half of: analyst_team.py](https://github.com/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/src/bootcamp_agent/projects/analyst_team.py)
- [Project 03, where the same team is measured](../03-analyst-team/notebook.ipynb)
- [Tutorial 1: calling a model from Python](01-calling-a-model.ipynb)
- [Tutorial 2: tools the model can call, with limits](02-tools-with-limits.ipynb)
- [LangGraph's own docs](https://langchain-ai.github.io/langgraph/)

## Ask your assistant about this tutorial

> **Ask your assistant.** Paste one of these into Claude Code, Cursor or any coding assistant, from the repo root.
>
> - "Explain how the four nodes in projects/tutorials/03-build-a-team-with-langgraph.ipynb become a graph, naming every LangGraph call. Do not change the code."
> - "Walk one question through the streamed run in section 5 and tell me where each model call happened. Do not change the code."
> - "I want to add a fifth role to this team. Ask me the three questions I should answer before I add the node."
> - "Explain what would break in this notebook if the calls field lost its reducer. Do not change the code."
> - "Compare this tutorial's parser with with_structured_output and tell me which one I should use in my capstone, and why."